# Follow-up run: the `m2_state` condition

Settings: **GPU T4 x2**, **Internet on**.

In the first run, `m2_choice` was identical to `m1_swap`: the follow-up pass recomputed every position without steering, so no trace of the internal push survived. `m2_state` keeps the steering on the question and answer positions (as a real KV cache would) and turns it off for the follow-up question and reply. Lower strengths look for a range where the push only sometimes flips the answer.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf llm-confabulation
!git clone --depth 1 https://github.com/pranavsaikosaraju/llm-confabulation.git
%cd llm-confabulation
!pip -q install -U "transformers>=4.56" accelerate 2>&1 | tail -n 2

## Smoke test (about 2 minutes)

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-0.5B-Instruct --out results/_smoke --n-per-task 20 --alphas 0.5 --only-m2-state
!head -c 1500 results/_smoke/transcripts_m2state.csv
!rm -rf results/_smoke

## Transcripts

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-1.5B-Instruct --out results/qwen2.5-1.5b-instruct --alphas 0.1,0.25,0.5,1 --only-m2-state

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-3B-Instruct --out results/qwen2.5-3b-instruct --alphas 0.1,0.25,0.5,1 --only-m2-state

In [ ]:
!python scripts/run_p2.py --model HuggingFaceTB/SmolLM2-1.7B-Instruct --out results/smollm2-1.7b-instruct --alphas 0.1,0.25,0.5,1 --only-m2-state

## Blind judge

In [ ]:
!python scripts/judge_p2.py --transcripts results/qwen2.5-1.5b-instruct/transcripts_m2state.csv --judge Qwen/Qwen2.5-7B-Instruct

In [ ]:
!python scripts/judge_p2.py --transcripts results/qwen2.5-3b-instruct/transcripts_m2state.csv --judge Qwen/Qwen2.5-7B-Instruct

In [ ]:
!python scripts/judge_p2.py --transcripts results/smollm2-1.7b-instruct/transcripts_m2state.csv --judge Qwen/Qwen2.5-7B-Instruct

In [ ]:
!cd results && zip -qr /kaggle/working/p2_m2state_results.zip . && ls -la /kaggle/working/p2_m2state_results.zip